# RAG Evaluation

Notebook này dùng để chạy benchmark RAG và xem kết quả theo cách gọn, dễ hiểu.

Có 4 cell:

1. Hướng dẫn.
2. Chạy benchmark thật qua API/RAG/Vector DB/LLM nếu bật `RUN_BENCHMARK = True`.
3. Kiểm tra input dataset.
4. Xem kết quả output bằng vài biểu đồ chính và bảng case cần sửa.

Mặc định notebook **không tự chạy benchmark** để tránh tốn API/LLM. Khi muốn chạy lại từ đầu, đổi `RUN_BENCHMARK = True` ở cell 2 rồi Run All.


## Step 1 - Có chạy lại benchmark không?

Cell tiếp theo là bước duy nhất có thể gọi hệ thống thật. Nếu `RUN_BENCHMARK = True`, notebook sẽ gọi backend, RAG chain, vector DB và LLM để tạo lại `docs/rag_benchmark_results.json`. Nếu chỉ muốn xem kết quả đã có, giữ `False`.


In [ ]:
# Cell 2 - Optional: run real RAG benchmark
# Set to True when you want to call API + RAG + Vector DB + LLM and regenerate results.
RUN_BENCHMARK = False

from pathlib import Path
import os
import subprocess
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "rag_benchmark_dataset.json").exists():
            return candidate
    raise FileNotFoundError("Cannot find docs/rag_benchmark_dataset.json from current directory")


ROOT = find_repo_root()
BACKEND_DIR = ROOT / "backend"
DATASET_PATH = ROOT / "docs" / "rag_benchmark_dataset.json"
RESULT_PATH = ROOT / "docs" / "rag_benchmark_results.json"

if RUN_BENCHMARK:
    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
    print("Running real RAG benchmark. This may call OpenRouter/LLM and Qdrant.")
    completed = subprocess.run(
        [sys.executable, "tests_local/test_rag_benchmark.py"],
        cwd=BACKEND_DIR,
        env=env,
        text=True,
        capture_output=True,
        timeout=900,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
else:
    print("Benchmark skipped. Set RUN_BENCHMARK = True to regenerate docs/rag_benchmark_results.json.")

print(f"Dataset: {DATASET_PATH.relative_to(ROOT)}")
print(f"Results: {RESULT_PATH.relative_to(ROOT)}")


## Step 2 - Kiểm tra bộ câu hỏi đầu vào

Cell tiếp theo kiểm tra golden dataset: có đủ cột cần thiết không, có trùng `id` không, và mỗi nhóm có bao nhiêu câu hỏi. Nếu dataset sai thì kết quả đánh giá RAG phía sau sẽ không đáng tin.


In [ ]:
# Cell 3 - Input check
import json

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)

with DATASET_PATH.open(encoding="utf-8") as f:
    dataset_df = pd.DataFrame(json.load(f))

required_columns = {"id", "group", "question", "ground_truth", "expected_source", "expected_behavior"}
missing_columns = required_columns - set(dataset_df.columns)
duplicate_ids = dataset_df[dataset_df.duplicated("id", keep=False)]

if missing_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")
if not duplicate_ids.empty:
    display(duplicate_ids[["id", "group", "question"]])
    raise ValueError("Dataset contains duplicate ids")

display(Markdown(f"## Input dataset OK: {len(dataset_df)} questions"))
display(dataset_df["group"].value_counts().rename_axis("group").reset_index(name="questions"))


## Step 3 - Đọc kết quả và kết luận nhanh

Cell cuối đọc output của benchmark, tính các điểm chính và chỉ giữ những biểu đồ dễ hiểu nhất. Bảng `Cases to inspect` là danh sách nên xem trước khi sửa RAG hoặc prompt.


In [ ]:
# Cell 4 - Output evaluation
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import matplotlib.pyplot as plt

LOW_SCORE_THRESHOLD = 0.70
CASE_ID = "FAQ-01"


def safe_mean(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna()
    return float(values.mean()) if not values.empty else 0.0


def normalize_sources(value) -> str:
    if isinstance(value, list):
        return ", ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def plot_bar(df: pd.DataFrame, x: str, y: str, title: str, ylabel: str, color: str):
    ax = df.plot(kind="bar", x=x, y=y, legend=False, color=color, figsize=(8, 4))
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, padding=3)
    plt.tight_layout()
    plt.show()


if not RESULT_PATH.exists():
    display(Markdown("## No benchmark results found\nRun cell 2 with `RUN_BENCHMARK = True` first."))
else:
    with RESULT_PATH.open(encoding="utf-8") as f:
        results_df = pd.DataFrame(json.load(f))

    non_knowledge_sources = {"user_context", "none", ""}
    results_df["source_ok_raw"] = results_df["source_ok"]
    results_df.loc[results_df["expected_source"].fillna("").isin(non_knowledge_sources), "source_ok"] = pd.NA

    avg_faithfulness = safe_mean(results_df["faithfulness"])
    avg_relevance = safe_mean(results_df["relevance"])
    source_recall = safe_mean(results_df["source_ok"])
    guardrail_rows = results_df[(results_df["group"] == "guardrail") & results_df["guardrail_pass"].notna()]
    guardrail_rate = safe_mean(guardrail_rows["guardrail_pass"]) if not guardrail_rows.empty else 0.0
    overall_score = 0.35 * avg_faithfulness + 0.25 * avg_relevance + 0.20 * source_recall + 0.20 * guardrail_rate

    scorecard = pd.DataFrame([
        {"metric": "Faithfulness", "score_pct": round(avg_faithfulness * 100, 2)},
        {"metric": "Relevance", "score_pct": round(avg_relevance * 100, 2)},
        {"metric": "Source recall", "score_pct": round(source_recall * 100, 2)},
        {"metric": "Guardrail", "score_pct": round(guardrail_rate * 100, 2)},
        {"metric": "Overall", "score_pct": round(overall_score * 100, 2)},
    ])

    group_summary = results_df.groupby("group").agg(
        cases=("id", "count"),
        faithfulness=("faithfulness", safe_mean),
        relevance=("relevance", safe_mean),
        source_recall=("source_ok", safe_mean),
        source_cases=("source_ok", "count"),
    ).reset_index()
    group_summary.loc[group_summary["source_cases"] == 0, "source_recall"] = pd.NA
    group_summary["quality_pct"] = ((group_summary["faithfulness"] + group_summary["relevance"]) / 2 * 100).round(2)
    group_summary["source_recall_display"] = group_summary["source_recall"].apply(lambda value: "N/A" if pd.isna(value) else f"{value * 100:.1f}%")

    failures_df = results_df[
        (pd.to_numeric(results_df["http_status"], errors="coerce") >= 400)
        | (pd.to_numeric(results_df["source_ok"], errors="coerce") == 0)
        | (pd.to_numeric(results_df["faithfulness"], errors="coerce") < LOW_SCORE_THRESHOLD)
        | (pd.to_numeric(results_df["relevance"], errors="coerce") < LOW_SCORE_THRESHOLD)
        | ((results_df["group"] == "guardrail") & (results_df["guardrail_pass"] == False))
    ].copy()
    failures_df["sources"] = failures_df["sources_returned"].apply(normalize_sources)

    display(Markdown(f"## RAG result summary: {len(results_df)} answers"))
    display(scorecard)
    plot_bar(scorecard, "metric", "score_pct", "Main RAG scores", "Percent", "#2563eb")

    display(Markdown("### Quality by question group"))
    display(group_summary[["group", "cases", "quality_pct", "source_recall_display"]])
    plot_bar(group_summary, "group", "quality_pct", "Answer quality by group", "Percent", "#059669")

    display(Markdown(f"### Cases to inspect: {len(failures_df)}"))
    display(failures_df[["id", "group", "question", "faithfulness", "relevance", "source_ok", "sources"]])

    matched = results_df[results_df["id"] == CASE_ID]
    if not matched.empty:
        row = matched.iloc[0]
        display(Markdown(f"### Detail for `{CASE_ID}`"))
        display(Markdown(f"**Question**\n\n{row['question']}"))
        display(Markdown(f"**Ground truth**\n\n{row['ground_truth']}"))
        display(Markdown(f"**Predicted**\n\n{row['predicted']}"))
